# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
print(con.sql(f"DESCRIBE SELECT * FROM '{fact_path}' LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. My rule and its reason codes
**Signal 1 — CTR vs. position tier: CONFIRMED.** Mean CTR declines from top_3 to deep (0.0038 → 0.0004), roughly a 9x drop, with n = 197K–508K in the largest buckets. Median CTR was 0 in every bucket because at this daily grain, 62–93% of rows per tier have zero clicks — so mean_ctr was used instead of median_ctr.

**Signal 2 — CTR vs. impression volume: MIXED.** Mean CTR is nearly flat across buckets (0.0028–0.0031), only a small, noisy decline at higher volume. Not strong enough to anchor a rule on its own.

**Rule (plain words):** Flag content items whose actual CTR is far below the mean CTR of their position tier, among rows with enough impressions (≥50) and usable GSC data (gsc_data_available IS TRUE).

**Reason codes:**
- `ctr_far_below_position_peers` — gap is large (threshold set in Section 2)
- `ctr_below_position_peers` — gap is moderate

**Action label:** `snippet_review`

In [3]:
q_signal = f"""
SELECT
    CASE
        WHEN gsc_sum_position / gsc_impressions <= 3 THEN 'top_3'
        WHEN gsc_sum_position / gsc_impressions <= 10 THEN 'page_1'
        WHEN gsc_sum_position / gsc_impressions <= 20 THEN 'page_2'
        WHEN gsc_sum_position / gsc_impressions <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    COUNT(*) AS n,
    ROUND(MEDIAN(gsc_clicks * 1.0 / gsc_impressions), 6) AS median_ctr,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr,
    ROUND(SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS zero_click_share
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY position_tier
ORDER BY MIN(gsc_sum_position / gsc_impressions)
"""
signal_result = con.sql(q_signal).df()
print(signal_result)

  position_tier       n  median_ctr  mean_ctr  zero_click_share
0         top_3  197354         0.0  0.003785            0.6213
1        page_1  507890         0.0  0.003347            0.6389
2        page_2  138253         0.0  0.003142            0.7072
3      page_3_5  184964         0.0  0.001566            0.7770
4          deep    8981         0.0  0.000408            0.9316


In [4]:
q_volume = f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'low_50_99'
        WHEN gsc_impressions < 500 THEN 'mid_100_499'
        WHEN gsc_impressions < 2000 THEN 'high_500_1999'
        ELSE 'very_high_2000plus'
    END AS volume_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY volume_bucket
ORDER BY MIN(gsc_impressions)
"""
volume_result = con.sql(q_volume).df()
print(volume_result)

        volume_bucket       n  mean_ctr
0           low_50_99  398834  0.003070
1         mid_100_499  537157  0.003101
2       high_500_1999   93794  0.002804
3  very_high_2000plus    7657  0.002805


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.